In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1"
CARPETA_DATASET = "CIC17__RUS_SMOTE_ENN__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ==== CONFIG SVM ====
C = 1.0
KERNEL = "rbf"      # "linear", "rbf", "poly", "sigmoid"
GAMMA = "scale"     # "scale", "auto" o un float

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__RUS_SMOTE_ENN__v1/CIC17__RUS_SMOTE_ENN__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__RUS_SMOTE_ENN__v1/CIC17__RUS_SMOTE_ENN__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(139345, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,52067,48,2,12,6,6,6.000000,0,0,250000.000000,...,360,-1,1,20,0.00000,0.00000,0,0,0.00000,0
1,53,23788,2,88,44,44,44.000000,104,104,12443.248700,...,-1,-1,1,32,0.00000,0.00000,0,0,0.00000,0
2,57234,47,1,0,0,0,0.000000,0,0,0.000000,...,114,247,0,32,0.00000,0.00000,0,0,0.00000,0
3,443,65202917,17,3727,1917,0,219.235294,1448,0,149.088422,...,29200,46,4,32,52496.66667,73129.77342,201772,22488,11574.76216,0
4,46494,97447839,7,11595,7240,0,1656.428571,373,0,123.060707,...,235,229,4,32,13954.00000,0.00000,13954,13954,0.00000,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,10000
14,10000
5,9914
12,9908
4,9903
9,9838
7,9809
3,9735
2,9729


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (139345, 47)
Shape y_train: (139345,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("svm", SVC(
        C=C,
        kernel=KERNEL,
        gamma=GAMMA,
        class_weight="balanced"
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'fu

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
    
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    
    "mcc": make_scorer(matthews_corrcoef)
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,fit_time,score_time
0,1,0.901001,0.908544,0.901001,0.892020,0.897817,0.882144,0.870285,0.894860,64.228109,58.297627
1,2,0.901396,0.909593,0.901396,0.892621,0.898945,0.882393,0.870787,0.895291,72.261794,45.992497
2,3,0.898884,0.906240,0.898884,0.889836,0.895293,0.879848,0.867856,0.892579,87.602259,49.428558
3,4,0.902113,0.910471,0.902113,0.893354,0.899694,0.883348,0.871520,0.896082,52.387103,53.279604
4,5,0.900032,0.907736,0.900032,0.890585,0.896561,0.880472,0.867913,0.893854,87.063512,48.835432


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_dataset": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_components_pca": N_COMPONENTS_PCA,
        "modelo": "SVM",
        "C": C,
        "kernel": KERNEL,
        "gamma": GAMMA,
        "dataset_balanceado": CARPETA_DATASET
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1',
 'dataset': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__RUS_SMOTE_ENN__v1/CIC17__RUS_SMOTE_ENN__v1__train.csv',
 'shape_dataset': {'rows': 139345, 'cols': 48},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_components_pca': 4,
  'modelo': 'SVM',
  'C': 1.0,
  'kernel': 'rbf',
  'gamma': 'scale',
  'dataset_balanceado': 'CIC17__RUS_SMOTE_ENN__v1'},
 'metricas_media': {'accuracy': 0.9006853493128565,
  'precision_weighted': 0.9085167473129607,
  'recall_weighted': 0.9006853493128565,
  'f1_weighted': 0.8916833244124114,
  'precision_macro': 0.8976622788917256,
  'recall_macro': 0.8816411259315041,
  'f1_macro': 0.8696722793395695,
  'mcc': 0.894533345728048,
  'fit_time': 72.7085551738739,
  'score_time': 51.16674365997314},
 'metricas_std': {'accuracy': 0.0012564906764127735,
  'precision_weighted': 0.0016410614165324408,
  'recall_weighted': 0.0012564906764

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.900685 ± 0.001256

Precision weighted  : 0.908517 ± 0.001641
Recall weighted     : 0.900685 ± 0.001256
F1 weighted         : 0.891683 ± 0.001449

Precision macro     : 0.897662 ± 0.001776
Recall macro        : 0.881641 ± 0.001442
F1 macro            : 0.869672 ± 0.001690

MCC                 : 0.894533 ± 0.001357

Fit time medio      : 72.708555
Score time medio    : 51.166744


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,fit_time,score_time
0,1,0.901001,0.908544,0.901001,0.892020,0.897817,0.882144,0.870285,0.894860,64.228109,58.297627
1,2,0.901396,0.909593,0.901396,0.892621,0.898945,0.882393,0.870787,0.895291,72.261794,45.992497
2,3,0.898884,0.906240,0.898884,0.889836,0.895293,0.879848,0.867856,0.892579,87.602259,49.428558
3,4,0.902113,0.910471,0.902113,0.893354,0.899694,0.883348,0.871520,0.896082,52.387103,53.279604
4,5,0.900032,0.907736,0.900032,0.890585,0.896561,0.880472,0.867913,0.893854,87.063512,48.835432


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(504160, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,443,93606787,18,3128,410,6,173.777778,1618,38,186.845426,...,256,8192,17,20,32068.8000,1749.725464,34107,30651,6.931172e+06,0
1,80,98331732,6,354,336,0,59.000000,4344,0,121.517233,...,0,235,3,20,21035.0000,0.000000,21035,21035,0.000000e+00,1
2,80,117175775,231,1427,401,0,6.177489,4584,0,5590.916723,...,29200,1039,4,32,206922.5455,636207.069100,2125159,14988,8.932177e+04,0
3,80,1285910,3,566,560,0,188.666667,1894,2,1923.929357,...,29200,15680,2,20,0.0000,0.000000,0,0,0.000000e+00,0
4,53,47941,1,44,44,44,44.000000,224,224,5590.204627,...,-1,-1,0,40,0.0000,0.000000,0,0,0.000000e+00,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,419012
1,34569
2,25603
3,18139
4,2057
5,1186
6,1077
7,1046
8,644


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (504160, 47)
Shape y_test: (504160,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
pipeline.fit(X_train, y_train)

print("Modelo final entrenado con todo el dataset train.")

Modelo final entrenado con todo el dataset train.


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

Predicciones en test generadas.
Número de predicciones: 504160


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.7254800063471913,
 'precision_weighted': 0.9449005770745621,
 'recall_weighted': 0.7254800063471913,
 'f1_weighted': 0.8022393635688089,
 'precision_macro': 0.28763335433110276,
 'recall_macro': 0.8474446929206759,
 'f1_macro': 0.3442350745480118,
 'mcc': 0.5503701268190512}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.725480

Precision weighted  : 0.944901
Recall weighted     : 0.725480
F1 weighted         : 0.802239

Precision macro     : 0.287633
Recall macro        : 0.847445
F1 macro            : 0.344235

MCC                 : 0.550370


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,284399,4845,14355,9902,7054,9007,2505,10460,10362,35342,2997,2417,6850,18514,3
1,89,32134,2107,0,119,0,34,6,0,23,0,1,16,40,0
2,19,508,25052,0,0,1,0,0,0,5,0,0,18,0,0
3,33,18,6,17950,0,0,15,6,0,72,9,0,29,1,0
4,20,6,37,0,1950,0,0,1,0,0,12,0,1,30,0
5,3,0,2,0,0,1178,0,0,0,0,0,0,0,3,0
6,21,0,1,0,14,21,972,29,0,0,1,1,0,17,0
7,13,0,0,0,0,0,8,1024,0,0,0,0,0,1,0
8,6,0,1,5,0,3,0,0,594,0,0,0,0,35,0
9,49,0,0,0,0,0,0,0,0,341,0,0,0,0,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========


              precision    recall  f1-score   support

           0       1.00      0.68      0.81    419012
           1       0.86      0.93      0.89     34569
           2       0.60      0.98      0.75     25603
           3       0.64      0.99      0.78     18139
           4       0.21      0.95      0.35      2057
           5       0.12      0.99      0.21      1186
           6       0.28      0.90      0.42      1077
           7       0.09      0.98      0.16      1046
           8       0.05      0.92      0.10       644
           9       0.01      0.87      0.02       390
          10       0.01      0.11      0.02       294
          11       0.04      0.95      0.08       130
          12       0.00      0.71      0.00         7
          13       0.00      0.75      0.00         4
          14       0.40      1.00      0.57         2

    accuracy                           0.73    504160
   macro avg       0.29      0.85      0.34    504160
weighted avg       0.94   

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_components_pca": N_COMPONENTS_PCA,
        "modelo": "SVM",
        "C": C,
        "kernel": KERNEL,
        "gamma": GAMMA,
        "dataset_balanceado": CARPETA_DATASET
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__RUS_SMOTE_ENN__v1/CIC17__RUS_SMOTE_ENN__v1__test.csv',
 'shape_test': {'rows': 504160, 'cols': 48},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_components_pca': 4,
  'modelo': 'SVM',
  'C': 1.0,
  'kernel': 'rbf',
  'gamma': 'scale',
  'dataset_balanceado': 'CIC17__RUS_SMOTE_ENN__v1'},
 'metricas_test': {'accuracy': 0.7254800063471913,
  'precision_weighted': 0.9449005770745621,
  'recall_weighted': 0.7254800063471913,
  'f1_weighted': 0.8022393635688089,
  'precision_macro': 0.28763335433110276,
  'recall_macro': 0.8474446929206759,
  'f1_macro': 0.3442350745480118,
  'mcc': 0.5503701268190512}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1/CIC17__RUS_SMOTE_ENN__v1__pca4_svm__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.9006853493128565, 'precision_weighted': 0.9085167473129607, 'recall_weighted': 0.9006853493128565, 'f1_weighted': 0.8916833244124114, 'precision_macro': 0.8976622788917256, 'recall_macro': 0.8816411259315041, 'f1_macro': 0.8696722793395695, 'mcc': 0.894533345728048, 'fit_time': 72.7085551738739, 'score_time': 51.16674365997314}

TEST:
{'accuracy': 0.7254800063471913, 'precision_weighted': 0.9449005770745621, 'recall_weighted': 0.7254800063471913, 'f1_weighted': 0.8022393635688089, 'precision_macro': 0.28763335433110276, 'recall_macro': 0.8474446929206759, 'f1_macro': 0.3442350745480118, 'mcc': 0.5503701268190512}
